<a href="https://colab.research.google.com/github/kgothatso-222/machine_leraning_internship_project1/blob/main/internship_Ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#importing the necessary libraries
import numpy as np
from sklearn.linear_model import LinearRegression

## You should already have `matplotlib`, `numpy` and `sklearn` installed and `ucimlrepo`. Type `pip install ucimlrepo`on your Terminal



In [2]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
student_performance = fetch_ucirepo(id=320)

# data (as pandas dataframes)
X = student_performance.data.features
y = student_performance.data.targets

# metadata
print(student_performance.metadata)

# variable information
print(student_performance.variables)


{'uci_id': 320, 'name': 'Student Performance', 'repository_url': 'https://archive.ics.uci.edu/dataset/320/student+performance', 'data_url': 'https://archive.ics.uci.edu/static/public/320/data.csv', 'abstract': 'Predict student performance in secondary education (high school). ', 'area': 'Social Science', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Multivariate'], 'num_instances': 649, 'num_features': 30, 'feature_types': ['Integer'], 'demographics': ['Sex', 'Age', 'Other', 'Education Level', 'Occupation'], 'target_col': ['G1', 'G2', 'G3'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2008, 'last_updated': 'Fri Jan 05 2024', 'dataset_doi': '10.24432/C5TG7T', 'creators': ['Paulo Cortez'], 'intro_paper': {'ID': 360, 'type': 'NATIVE', 'title': 'Using data mining to predict secondary school student performance', 'authors': 'P. Cortez, A. M. G. Silva', 'venue': 'Proceedings of 5th Annual Future Business Technolo

# Predict students' final academic performance from historical academic records

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd

# Selecting G1 and G2 as the historical academic records (features) to predict G3 (target)
historical_grades = y[['G1', 'G2']]
final_grade = y['G3']


X_train, X_test, y_train, y_test = train_test_split(
    historical_grades, final_grade, test_size=0.2, random_state=42
)

# Initialize and train the Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Model Coefficients (for G1, G2): {model.coef_}")
print(f"Intercept: {model.intercept_:.4f}")
print(f"Mean Squared Error: {mse:.4f}")
print(f"R-squared Score: {r2:.4f}")

Model Coefficients (for G1, G2): [0.1881616  0.88218903]
Intercept: -0.4313
Mean Squared Error: 1.3663
R-squared Score: 0.8599


In [4]:
#  actual vs predicted results side by side for a quick preview
results_df = pd.DataFrame({'Actual G3': y_test, 'Predicted G3': y_pred.round(2)})
display(results_df.head(10))

,Actual G3,Predicted G3
636,19,18.65
220,12,11.34
594,18,18.83
429,11,11.15
72,11,11.72
448,17,16.69
181,18,17.76
131,8,9.39
231,10,10.27
277,11,10.46


### Evaluating Multi-variable Features
Now we will evaluate the impact of demographic and social variables (`Medu`, `Fedu`, `studytime`, and `absences`) alongside prior grades (`G1`, `G2`) to predict the final grade `G3`.

In [5]:
# Select additional features from X and combine them with the historical grades from y
multi_features = pd.DataFrame({
    'G1': y['G1'],
    'G2': y['G2'],
    'Medu': X['Medu'],
    'Fedu': X['Fedu'],
    'studytime': X['studytime'],
    'absences': X['absences']
})

# Split the dataset using the same random state for comparative consistency
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    multi_features, final_grade, test_size=0.2, random_state=42
)
multi_model = LinearRegression()
multi_model.fit(X_train_multi, y_train_multi)
y_pred_multi = multi_model.predict(X_test_multi)
mse_multi = mean_squared_error(y_test_multi, y_pred_multi)
r2_multi = r2_score(y_test_multi, y_pred_multi)

# Display coefficients alongside feature names
coefficients = pd.DataFrame({
    'Feature': multi_features.columns,
    'Coefficient': multi_model.coef_
})

print(f"Multi-Variable Model Intercept: {multi_model.intercept_:.4f}\n")
print("Feature Coefficients:")
display(coefficients)

print(f"\n--- Comparison of Results ---")
print(f"[Previous Model (G1 & G2 only)] MSE: {mse:.4f} | R2: {r2:.4f}")
print(f"[Multi-Variable Model]         MSE: {mse_multi:.4f} | R2: {r2_multi:.4f}")

Multi-Variable Model Intercept: -0.6522

Feature Coefficients:


,Feature,Coefficient
0,G1,0.190647
1,G2,0.883760
2,Medu,-0.075718
3,Fedu,0.018366
4,studytime,0.121222
5,absences,0.024662



--- Comparison of Results ---
[Previous Model (G1 & G2 only)] MSE: 1.3663 | R2: 0.8599
[Multi-Variable Model]         MSE: 1.3953 | R2: 0.8569


## Implement Linear Regression, Ridge, and Decision Tree regressors with R² and RMSE scoring.

In [6]:
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor


#  Prepare data
X_features = multi_features
y_target = final_grade

X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_target, test_size=0.2, random_state=42
)

# The models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42)
}

# Evaluating each model using R² and RMSE
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)
    results[name] = {'RMSE': rmse, 'R2': r2, 'Model': model}

# Print model performance comparison
performance_df = pd.DataFrame(results).T[['R2', 'RMSE']]
display(performance_df)

,R2,RMSE
Linear Regression,0.856916,1.181233
Ridge Regression,0.856922,1.181208
Decision Tree,0.840366,1.247682


# Identifying students who may need additional academic support and help teachers improve learning outcomes.

In [7]:

best_model = results['Linear Regression']['Model']
test_predictions = best_model.predict(X_test)

# DataFrame with student details to identify at-risk students
support_df = X_test.copy()
support_df['Actual_G3'] = y_test
support_df['Predicted_G3'] = test_predictions.round(2)

# Filter students predicted to score below 10
at_risk_students = support_df[support_df['Predicted_G3'] < 10]

print(f"Total students in test set: {len(X_test)}")
print(f"Number of students identified as needing academic support: {len(at_risk_students)}")
print("\nPreview of identified students requiring intervention:")
display(at_risk_students.sort_values(by='Predicted_G3'))

Total students in test set: 130
Number of students identified as needing academic support: 27

Preview of identified students requiring intervention:


,G1,G2,Medu,Fedu,studytime,absences,Actual_G3,Predicted_G3
603,5,0,4,2,2,0,0,0.28
518,8,5,2,4,2,8,8,5.65
514,7,6,3,2,1,2,7,5.97
478,7,7,1,1,2,0,8,7.05
585,8,7,3,1,3,5,8,7.34
570,7,8,2,2,1,6,8,7.91
522,8,8,4,4,2,4,8,8.05
174,8,8,2,1,1,8,9,8.13
601,8,8,2,2,2,4,10,8.17
177,9,8,1,3,1,2,8,8.28
